# LeetCode 1593 — Split a String Into the Max Number of Unique Substrings bằng Python

Link đề: https://leetcode.com/problems/split-a-string-into-the-max-number-of-unique-substrings/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Backtracking

Skeleton LeetCode:

```python
class Solution:
    def maxUniqueSplit(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu rằng một cách cắt chuỗi độ dài `n` tương ứng với một **tập vị trí cắt** trong `n − 1` khe ⇒ có đúng `2ⁿ⁻¹` cách cắt; liệt kê chúng bằng **bitmask**.
2. Viết **quay lui (backtracking)** chuẩn: chọn → đệ quy → **hoàn tác (undo)**, dùng `set` để kiểm tra trùng trong O(1).
3. Thêm **cắt tỉa (pruning)** bằng cận trên "mỗi ký tự còn lại tối đa thành 1 đoạn" và đo số lời gọi đệ quy để thấy hiệu quả.
4. Nhận ra vì sao **tham lam (greedy)** và **quy hoạch động (DP)** thông thường không áp dụng được ở đây (trạng thái phụ thuộc cả tập đã dùng).
5. Tự kiểm thử bằng vét cạn trên mọi chuỗi nhỏ của bảng chữ cái `{a, b}` / `{a, b, c}`.

## 2. Đề bài & ví dụ

Cho chuỗi `s`. Hãy cắt `s` thành một danh sách các **chuỗi con khác rỗng** sao cho nối lại theo thứ tự ta được đúng `s`, và **mọi chuỗi con trong danh sách đều đôi một khác nhau**. Trả về **số lượng chuỗi con lớn nhất** có thể đạt được.

*Chuỗi con (substring)* là dãy ký tự **liên tiếp** trong chuỗi.

**Ví dụ 1:** `s = "ababccc"` → `5`. Một cách tối ưu: `['a', 'b', 'ab', 'c', 'cc']`. Cách `['a', 'b', 'a', 'b', 'c', 'cc']` không hợp lệ vì `'a'` và `'b'` xuất hiện nhiều lần.

**Ví dụ 2:** `s = "aba"` → `2`. Ví dụ `['a', 'ba']`.

**Ví dụ 3:** `s = "aa"` → `1`. Không thể cắt thêm (`['a','a']` bị trùng).

**Ràng buộc:**
- `1 ≤ s.length ≤ 16`
- `s` chỉ gồm chữ cái tiếng Anh thường.

Ràng buộc `n ≤ 16` là **tín hiệu mạnh**: đề muốn ta vét cạn / quay lui (vì `2¹⁵ = 32768` là rất nhỏ).

In [1]:
import random
import time
import itertools
from typing import List

examples = [("ababccc", 5), ("aba", 2), ("aa", 1)]
for s, want in examples:
    print(f"s={s!r:12} n={len(s):2d}  số cách cắt = 2^(n-1) = {2 ** (len(s) - 1):5d}  -> mong đợi {want}")

s='ababccc'    n= 7  số cách cắt = 2^(n-1) =    64  -> mong đợi 5
s='aba'        n= 3  số cách cắt = 2^(n-1) =     4  -> mong đợi 2
s='aa'         n= 2  số cách cắt = 2^(n-1) =     2  -> mong đợi 1


## 3. Kiến thức nền

### 3.1 Cắt chuỗi (slicing)

`s[i:j]` lấy các ký tự từ chỉ số `i` đến `j − 1`. Với `0 ≤ i < j ≤ n`, có `n(n+1)/2` chuỗi con (tính theo vị trí).

In [2]:
s = "ababccc"
print(s[0:1], s[2:4], s[4:7])
print("mọi tiền tố bắt đầu tại 2:", [s[2:j] for j in range(3, len(s) + 1)])

a ab ccc
mọi tiền tố bắt đầu tại 2: ['a', 'ab', 'abc', 'abcc', 'abccc']


### 3.2 `set` — thêm, xóa, kiểm tra thuộc

`set` lưu các phần tử không trùng; `x in st`, `st.add(x)`, `st.remove(x)` đều trung bình O(1) (với chuỗi: O(độ dài) để băm). Trong quay lui, `add` trước khi đi sâu và `remove` khi quay về là cặp **bắt buộc đi đôi**.

In [3]:
used = set()
used.add("a"); used.add("ab")
print("a" in used, "b" in used, used)
used.remove("a")
print(used)
print("độ dài list vs set:", len(["a", "b", "a"]), len(set(["a", "b", "a"])))

True False {'ab', 'a'}
{'ab'}
độ dài list vs set: 3 2


### 3.3 Bitmask liệt kê tập con

Một số nguyên `mask` trong `[0, 2^(n−1))` có `n − 1` bit; bit `i` bằng 1 nghĩa là "cắt sau ký tự thứ `i`". `mask >> i & 1` đọc bit `i`, `bin(mask).count("1")` (hoặc `mask.bit_count()` từ Python 3.10) đếm số bit 1 ⇒ số đoạn = số lát cắt + 1.

In [4]:
def split_by_mask(s, mask):
    parts, start = [], 0
    for i in range(len(s) - 1):
        if mask >> i & 1:
            parts.append(s[start:i + 1])
            start = i + 1
    parts.append(s[start:])
    return parts

for mask in range(4):
    print(f"mask={mask:02b} -> {split_by_mask('aba', mask)}")

mask=00 -> ['aba']
mask=01 -> ['a', 'ba']
mask=10 -> ['ab', 'a']
mask=11 -> ['a', 'b', 'a']


### 3.4 Đệ quy và biến `nonlocal`

Hàm lồng bên trong muốn **gán lại** biến của hàm ngoài (ví dụ `best`) phải khai báo `nonlocal best`. Nếu chỉ **đọc** hoặc gọi phương thức (như `used.add`) thì không cần.

In [5]:
def demo():
    best = 0
    def go(k):
        nonlocal best
        best = max(best, k)
        if k < 3:
            go(k + 1)
    go(0)
    return best
print(demo())

3


## 4. Ý tưởng nền tảng / trực giác

Mỗi đoạn càng ngắn thì càng nhiều đoạn — nhưng đoạn ngắn dễ **trùng**. Ta phải cân bằng: đôi khi chấp nhận một đoạn dài hơn ở đầu để dành các chữ cái ngắn cho phía sau.

**Tính tay `s = "aba"`** (4 cách cắt):

| mask | các đoạn | trùng? | số đoạn |
|---|---|---|---|
| 00 | aba | không | 1 |
| 01 | a, ba | không | **2** |
| 10 | ab, a | không | **2** |
| 11 | a, b, a | **có** ('a') | — |

⇒ đáp án 2.

**Tính tay `s = "aaba"`:** cắt `a | ab | a` bị trùng `'a'`; nhưng `aa | b | a` gồm 3 đoạn khác nhau ⇒ 3. Lưu ý đoạn đầu dài `"aa"` lại giúp ta được nhiều đoạn hơn — gợi ý rằng "lấy đoạn ngắn nhất có thể" (tham lam) **không** an toàn.

In [6]:
for s in ["aba", "aaba"]:
    n = len(s)
    for mask in range(1 << (n - 1)):
        parts = split_by_mask(s, mask)
        ok = len(set(parts)) == len(parts)
        print(f"{s}: mask={mask:0{n-1}b} {parts!s:22} {'hợp lệ, ' + str(len(parts)) + ' đoạn' if ok else 'trùng'}")
    print()

aba: mask=00 ['aba']                hợp lệ, 1 đoạn
aba: mask=01 ['a', 'ba']            hợp lệ, 2 đoạn
aba: mask=10 ['ab', 'a']            hợp lệ, 2 đoạn
aba: mask=11 ['a', 'b', 'a']        trùng

aaba: mask=000 ['aaba']               hợp lệ, 1 đoạn
aaba: mask=001 ['a', 'aba']           hợp lệ, 2 đoạn
aaba: mask=010 ['aa', 'ba']           hợp lệ, 2 đoạn
aaba: mask=011 ['a', 'a', 'ba']       trùng
aaba: mask=100 ['aab', 'a']           hợp lệ, 2 đoạn
aaba: mask=101 ['a', 'ab', 'a']       trùng
aaba: mask=110 ['aa', 'b', 'a']       hợp lệ, 3 đoạn
aaba: mask=111 ['a', 'a', 'b', 'a']   trùng



## 5. Cách 1 — Brute Force / baseline: duyệt mọi bitmask

**Ý tưởng:** duyệt mọi `mask` trong `[0, 2^(n−1))`, cắt chuỗi theo mask, kiểm tra `len(set(parts)) == len(parts)`; giữ số đoạn lớn nhất.

In [7]:
def max_split_bitmask(s):
    n = len(s)
    best = 1                         # không cắt gì luôn hợp lệ
    for mask in range(1 << (n - 1)):
        parts = split_by_mask(s, mask)
        if len(parts) > best and len(set(parts)) == len(parts):
            best = len(parts)
    return best

for s, want in examples:
    print(s, "->", max_split_bitmask(s), "| mong đợi", want)

ababccc -> 5 | mong đợi 5
aba -> 2 | mong đợi 2
aa -> 1 | mong đợi 1


**Độ phức tạp:** `2ⁿ⁻¹` mask, mỗi mask cắt và băm tốn O(n) ⇒ **O(n · 2ⁿ)** thời gian, O(n) bộ nhớ. Với `n = 16` là ~`5 · 10⁵` thao tác — vẫn chấp nhận được.

**Nút thắt:** ta luôn làm đủ `2ⁿ⁻¹` mask, kể cả những mask mà ngay đoạn đầu đã trùng — không "cắt nhánh" sớm được.

In [8]:
s16 = "abcdefghijklmnop"
t = time.perf_counter()
print(max_split_bitmask(s16), f"{time.perf_counter() - t:.3f}s  (n=16, mọi ký tự khác nhau)")

16 0.035s  (n=16, mọi ký tự khác nhau)


## 6. Ý tưởng thuật toán / bước đột phá

Thay vì sinh xong cả cách cắt rồi mới kiểm tra, ta **dựng dần từ trái sang phải**: ở vị trí `start`, thử mọi đoạn tiếp theo `s[start:end]`. **Nếu đoạn đó đã dùng thì bỏ ngay** — toàn bộ các cách cắt có chung tiền tố đó bị loại một lần (đó là "cắt nhánh tự nhiên" của quay lui).

**Quy tắc:**
- Nếu `start == n` thì ta có một cách cắt hợp lệ với `count` đoạn ⇒ cập nhật `best`.
- Nếu `s[start:end]` chưa có trong `used` thì: thêm vào → đệ quy từ `end` với `count + 1` → **xóa khỏi `used`**.
- **Cắt tỉa:** từ vị trí `start` còn `n − start` ký tự, mỗi đoạn dài ≥ 1 nên tối đa thêm `n − start` đoạn. Nếu `count + (n − start) ≤ best` thì nhánh này **không thể** vượt `best` ⇒ quay về ngay.

**Invariant:** khi vào `dfs(start, count)`, `used` chứa đúng `count` đoạn đã chọn, ghép lại bằng `s[:start]`.

**Vì sao không DP được?** Kết quả của phần đuôi `s[start:]` phụ thuộc vào **tập** đoạn đã dùng ở phần đầu (một tập bất kỳ các chuỗi), nên không có trạng thái nhỏ gọn để ghi nhớ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quay lui với `set` (chưa cắt tỉa)

Từng đoạn:
1. `used = set()`, `best = 0`.
2. `dfs(start, count)`: nếu `start == n` cập nhật `best`.
3. Vòng `end` từ `start + 1` đến `n`: lấy `piece = s[start:end]`; nếu chưa dùng thì `add` → `dfs(end, count + 1)` → `remove`.

Ta đếm thêm số lời gọi `dfs` để so sánh với Cách 3.

In [9]:
def max_split_backtrack(s, stats=None):
    n = len(s)
    used = set()
    best = 0
    calls = 0

    def dfs(start, count):
        nonlocal best, calls
        calls += 1
        if start == n:
            best = max(best, count)
            return
        for end in range(start + 1, n + 1):
            piece = s[start:end]
            if piece not in used:
                used.add(piece)
                dfs(end, count + 1)
                used.remove(piece)     # hoàn tác

    dfs(0, 0)
    if stats is not None:
        stats["calls"] = calls
    return best

for s, want in examples:
    st = {}
    print(s, "->", max_split_backtrack(s, st), "| mong đợi", want, "| số lời gọi", st["calls"])

ababccc -> 5 | mong đợi 5 | số lời gọi 85
aba -> 2 | mong đợi 2 | số lời gọi 7
aa -> 1 | mong đợi 1 | số lời gọi 3


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def maxUniqueSplit(self, s: str) -> int:
        n = len(s)
        used = set()
        best = 0
        def dfs(start, count):
            nonlocal best
            if start == n:
                best = max(best, count)
                return
            for end in range(start + 1, n + 1):
                piece = s[start:end]
                if piece not in used:
                    used.add(piece)
                    dfs(end, count + 1)
                    used.remove(piece)
        dfs(0, 0)
        return best
```

**Độ phức tạp:** worst case vẫn O(n · 2ⁿ) (chuỗi toàn ký tự khác nhau: mọi nhánh đều hợp lệ), nhưng thường nhanh hơn nhiều vì các nhánh có đoạn trùng bị loại sớm. Bộ nhớ O(n) cho ngăn xếp đệ quy + `set` (tổng độ dài các đoạn ≤ n).

### 7.2 Cách 3 — Quay lui + cắt tỉa cận trên

Thêm đúng **một dòng** ở đầu `dfs`:

```python
if count + (n - start) <= best:
    return
```

Lý do: số đoạn tối đa có thể đạt từ đây là `count + (n − start)` (mỗi ký tự còn lại thành một đoạn — lạc quan nhất). Nếu ngay cả trường hợp lạc quan nhất cũng không **vượt** `best` thì bỏ nhánh.

Mẹo phụ: duyệt `end` từ **nhỏ đến lớn** (đoạn ngắn trước) giúp tìm sớm một đáp án tốt ⇒ `best` lớn sớm ⇒ cắt tỉa mạnh hơn. Và khi `best == n` (mọi ký tự thành một đoạn) thì đã là tối đa tuyệt đối.

In [10]:
def max_split_pruned(s, stats=None):
    n = len(s)
    used = set()
    best = 0
    calls = 0

    def dfs(start, count):
        nonlocal best, calls
        calls += 1
        if count + (n - start) <= best:   # cắt tỉa
            return
        if start == n:
            best = count                  # đến đây chắc chắn count > best
            return
        for end in range(start + 1, n + 1):
            piece = s[start:end]
            if piece not in used:
                used.add(piece)
                dfs(end, count + 1)
                used.remove(piece)

    dfs(0, 0)
    if stats is not None:
        stats["calls"] = calls
    return best

for s in ["ababccc", "aaaaaaaaaaaaaaaa", "abcdefghijklmnop", "aabbccddeeffgghh"]:
    a, b = {}, {}
    r1 = max_split_backtrack(s, a); r2 = max_split_pruned(s, b)
    print(f"{s:18} kết quả {r1}/{r2} | lời gọi: không tỉa {a['calls']:6d}, có tỉa {b['calls']:5d}")

ababccc            kết quả 5/5 | lời gọi: không tỉa     85, có tỉa    42
aaaaaaaaaaaaaaaa   kết quả 5/5 | lời gọi: không tỉa   1419, có tỉa  1215
abcdefghijklmnop   kết quả 16/16 | lời gọi: không tỉa  65536, có tỉa   137
aabbccddeeffgghh   kết quả 12/12 | lời gọi: không tỉa  22363, có tỉa   753


Bảng trên cho thấy cắt tỉa giảm số lời gọi rất mạnh, đặc biệt với chuỗi toàn ký tự khác nhau (`abcd…`): nhánh đầu tiên đã cho `best = 16 = n`, mọi nhánh khác bị cắt ngay.

Code nộp LeetCode của Cách 3 (xem mục 9).

**Độ phức tạp:** worst case lý thuyết vẫn là hàm mũ, nhưng thực tế nhanh hơn Cách 2 nhiều lần. Bộ nhớ O(n).

### 7.3 Cách 4 — Bitmask duyệt theo số đoạn giảm dần (dừng sớm)

Một cách nghĩ khác cho brute force: muốn **nhiều đoạn nhất** thì thử các mask có **nhiều bit 1 nhất trước**. Mask đầu tiên hợp lệ ⇒ trả ngay số đoạn của nó. Ta nhóm mask theo `popcount` từ `n − 1` xuống `0`.

Ưu: code ngắn, không đệ quy. Nhược: phải sắp xếp/nhóm `2ⁿ⁻¹` mask trước (O(2ⁿ) bộ nhớ), và worst case (đáp án nhỏ, ví dụ `"aaaa…"`) vẫn duyệt gần hết.

In [11]:
def max_split_by_popcount(s):
    n = len(s)
    groups = [[] for _ in range(n)]          # groups[c] = các mask có c lát cắt
    for mask in range(1 << (n - 1)):
        groups[bin(mask).count("1")].append(mask)
    for cuts in range(n - 1, -1, -1):
        for mask in groups[cuts]:
            parts = split_by_mask(s, mask)
            if len(set(parts)) == len(parts):
                return cuts + 1
    return 1

for s, want in examples:
    print(s, "->", max_split_by_popcount(s), "| mong đợi", want)

ababccc -> 5 | mong đợi 5
aba -> 2 | mong đợi 2
aa -> 1 | mong đợi 1


Code nộp LeetCode của Cách 4:

```python
class Solution:
    def maxUniqueSplit(self, s: str) -> int:
        n = len(s)
        groups = [[] for _ in range(n)]
        for mask in range(1 << (n - 1)):
            groups[bin(mask).count("1")].append(mask)
        for cuts in range(n - 1, -1, -1):
            for mask in groups[cuts]:
                parts, start = [], 0
                for i in range(n - 1):
                    if mask >> i & 1:
                        parts.append(s[start:i + 1])
                        start = i + 1
                parts.append(s[start:])
                if len(set(parts)) == len(parts):
                    return cuts + 1
        return 1
```

**Độ phức tạp:** Time O(n · 2ⁿ) worst case, Space O(2ⁿ) cho các nhóm.

## 8. TRACE từng bước

Trace Cách 3 (có cắt tỉa) trên `s = "aba"` (`n = 3`):

| bước | start | count | used | thử piece | hành động | best |
|---|---|---|---|---|---|---|
| 1 | 0 | 0 | {} | 'a' | thêm, đi sâu | 0 |
| 2 | 1 | 1 | {a} | 'b' | thêm, đi sâu | 0 |
| 3 | 2 | 2 | {a,b} | 'a' | đã dùng → bỏ | 0 |
| 4 | 2 | 2 | {a,b} | — | hết lựa chọn, về | 0 |
| 5 | 1 | 1 | {a} | 'ba' | thêm, đi sâu → start=3 | **2** |
| 6 | 0 | 0 | {} | 'ab' | tỉa? 1+(3−2)=2 ≤ 2 → cắt | 2 |
| 7 | 0 | 0 | {} | 'aba' | tỉa? 1+0=1 ≤ 2 → cắt | 2 |

Kết quả 2.

In [12]:
def trace_pruned(s):
    n = len(s)
    used = []
    best = 0

    def dfs(start, count, depth):
        nonlocal best
        pad = "  " * depth
        if count + (n - start) <= best:
            print(f"{pad}dfs(start={start}, count={count}) cận trên {count + n - start} <= best={best} -> CẮT TỈA")
            return
        if start == n:
            best = count
            print(f"{pad}dfs(start={start}) HOÀN THÀNH {used} -> best={best}")
            return
        print(f"{pad}dfs(start={start}, count={count}) used={used}")
        for end in range(start + 1, n + 1):
            piece = s[start:end]
            if piece in used:
                print(f"{pad}  thử {piece!r}: đã dùng -> bỏ")
                continue
            print(f"{pad}  thử {piece!r}: thêm")
            used.append(piece)
            dfs(end, count + 1, depth + 1)
            used.pop()
    dfs(0, 0, 0)
    print("=> kết quả", best)
    return best

trace_pruned("aba")

dfs(start=0, count=0) used=[]
  thử 'a': thêm
  dfs(start=1, count=1) used=['a']
    thử 'b': thêm
    dfs(start=2, count=2) used=['a', 'b']
      thử 'a': đã dùng -> bỏ
    thử 'ba': thêm
    dfs(start=3) HOÀN THÀNH ['a', 'ba'] -> best=2
  thử 'ab': thêm
  dfs(start=2, count=1) cận trên 2 <= best=2 -> CẮT TỈA
  thử 'aba': thêm
  dfs(start=3, count=1) cận trên 1 <= best=2 -> CẮT TỈA
=> kết quả 2


2

In [13]:
trace_pruned("aaba")

dfs(start=0, count=0) used=[]
  thử 'a': thêm
  dfs(start=1, count=1) used=['a']
    thử 'a': đã dùng -> bỏ
    thử 'ab': thêm
    dfs(start=3, count=2) used=['a', 'ab']
      thử 'a': đã dùng -> bỏ
    thử 'aba': thêm
    dfs(start=4) HOÀN THÀNH ['a', 'aba'] -> best=2
  thử 'aa': thêm
  dfs(start=2, count=1) used=['aa']
    thử 'b': thêm
    dfs(start=3, count=2) used=['aa', 'b']
      thử 'a': thêm
      dfs(start=4) HOÀN THÀNH ['aa', 'b', 'a'] -> best=3
    thử 'ba': thêm
    dfs(start=4, count=2) cận trên 2 <= best=3 -> CẮT TỈA
  thử 'aab': thêm
  dfs(start=3, count=1) cận trên 2 <= best=3 -> CẮT TỈA
  thử 'aaba': thêm
  dfs(start=4, count=1) cận trên 1 <= best=3 -> CẮT TỈA
=> kết quả 3


3

Trong trace `"aaba"`: nhánh `a | ab | …` bị kẹt vì `'a'` cuối trùng; nhánh `a | aba` chỉ được 2; nhánh `aa | b | a` cho 3 — tìm được nhờ quay lui thử cả đoạn đầu dài hơn.

## 9. Code nộp LeetCode

```python
class Solution:
    def maxUniqueSplit(self, s: str) -> int:
        n = len(s)
        used = set()
        best = 0

        def dfs(start: int, count: int) -> None:
            nonlocal best
            if count + (n - start) <= best:      # cắt tỉa cận trên
                return
            if start == n:
                best = count
                return
            for end in range(start + 1, n + 1):
                piece = s[start:end]
                if piece not in used:
                    used.add(piece)
                    dfs(end, count + 1)
                    used.remove(piece)

        dfs(0, 0)
        return best
```

In [14]:
class Solution:
    def maxUniqueSplit(self, s: str) -> int:
        n = len(s)
        used = set()
        best = 0

        def dfs(start: int, count: int) -> None:
            nonlocal best
            if count + (n - start) <= best:
                return
            if start == n:
                best = count
                return
            for end in range(start + 1, n + 1):
                piece = s[start:end]
                if piece not in used:
                    used.add(piece)
                    dfs(end, count + 1)
                    used.remove(piece)

        dfs(0, 0)
        return best

sol = Solution()
for s, want in examples:
    got = sol.maxUniqueSplit(s)
    print(s, "->", got, "OK" if got == want else "SAI")

ababccc -> 5 OK
aba -> 2 OK
aa -> 1 OK


## 10. Vì sao thuật toán đúng

1. **Quay lui duyệt đủ:** mọi cách cắt hợp lệ là một dãy đoạn `p₁, p₂, …, p_k` khác nhau. Quay lui chọn `p₁` trong mọi tiền tố của `s`, rồi `p₂` trong mọi tiền tố của phần còn lại, … Một nhánh chỉ bị loại khi đoạn đang thử **đã dùng** — và khi đó mọi cách cắt kéo dài nhánh đó đều có trùng, nên không mất cách hợp lệ nào.
2. **Hoàn tác giữ invariant:** nhờ `remove` sau khi đệ quy về, `used` luôn đúng bằng tập đoạn trên đường đi hiện tại; các nhánh anh em không bị ảnh hưởng bởi nhau.
3. **Cắt tỉa an toàn:** từ trạng thái `(start, count)`, mọi cách hoàn thành có thêm tối đa `n − start` đoạn. Nếu `count + n − start ≤ best` thì không nhánh con nào cho kết quả **lớn hơn** `best` — bỏ nó không làm mất đáp án tối ưu (`best` đã là một giá trị đạt được).
4. Khi `start == n` mà chưa bị tỉa thì `count > best` ⇒ gán `best = count` là đúng.

## 11. Phân tích Time / Space

Gọi `n = len(s)` (≤ 16).

- **Cách 1 (bitmask):** `2ⁿ⁻¹` mask × O(n) mỗi mask ⇒ O(n · 2ⁿ). Space O(n).
- **Cách 2 (quay lui):** số nút cây đệ quy ≤ số tiền tố của các cách cắt ≤ `2ⁿ`; mỗi nút tốn O(n) (cắt chuỗi + băm) cho mỗi lựa chọn ⇒ worst O(n² · 2ⁿ) lỏng, thực tế O(n · 2ⁿ). Space O(n) (độ sâu đệ quy ≤ n, `set` chứa tổng độ dài ≤ n).
- **Cách 3 (có tỉa):** cùng worst case lý thuyết, nhưng hằng số nhỏ hơn rất nhiều.
- **Cách 4:** O(n · 2ⁿ) time, O(2ⁿ) space.

Đo thực tế với các chuỗi `n = 16`:

In [15]:
random.seed(11)
tests = ["abcdefghijklmnop", "aaaaaaaaaaaaaaaa", "abababababababab",
         "".join(random.choice("ab") for _ in range(16)),
         "".join(random.choice("abc") for _ in range(16))]
for s in tests:
    row = []
    for f in [max_split_bitmask, max_split_backtrack, max_split_pruned, max_split_by_popcount]:
        t = time.perf_counter(); r = f(s); row.append(f"{r:2d} {time.perf_counter() - t:.3f}s")
    print(f"{s}: " + " | ".join(row))

abcdefghijklmnop: 16 0.042s | 16 0.016s | 16 0.000s | 16 0.004s


aaaaaaaaaaaaaaaa:  5 0.045s |  5 0.000s |  5 0.000s |  5 0.047s


abababababababab:  7 0.044s |  7 0.002s |  7 0.001s |  7 0.057s


bbbaabaabbaaabba:  8 0.075s |  8 0.003s |  8 0.001s |  8 0.050s
cacaaaaacabbbcac:  9 0.049s |  9 0.003s |  9 0.001s |  9 0.022s


## 12. Bộ test

Đối chiếu 4 cách và lớp `Solution` trên: ví dụ đề, edge case, và **mọi** chuỗi trên `{a,b}` độ dài 1..10, mọi chuỗi trên `{a,b,c}` độ dài 1..7, cùng chuỗi ngẫu nhiên độ dài tới 16.

In [16]:
ALL = [max_split_bitmask, max_split_backtrack, max_split_pruned,
       max_split_by_popcount, lambda x: Solution().maxUniqueSplit(x)]

fixed = examples + [
    ("a", 1), ("ab", 2), ("aaaa", 2), ("aaba", 3), ("abcdefghijklmnop", 16),
    ("aaaaaaaaaaaaaaaa", 5),   # a, aa, aaa, aaaa, aaaaaa? -> kiểm chứng bên dưới
]
for s, w in fixed:
    truth = max_split_bitmask(s)
    assert truth == w, (s, truth, w)
    for f in ALL:
        assert f(s) == w, (f, s)
print("Ca cố định: OK")

cnt = 0
for L in range(1, 11):
    for tup in itertools.product("ab", repeat=L):
        s = "".join(tup); want = max_split_pruned(s)
        assert max_split_bitmask(s) == want == max_split_backtrack(s)
        cnt += 1
for L in range(1, 8):
    for tup in itertools.product("abc", repeat=L):
        s = "".join(tup); want = max_split_bitmask(s)
        assert max_split_pruned(s) == want == max_split_by_popcount(s)
        cnt += 1
print("Vét cạn", cnt, "chuỗi nhỏ: khớp")

Ca cố định: OK


Vét cạn 5325 chuỗi nhỏ: khớp


Về ca `"a" * 16`: các đoạn phải có độ dài đôi một khác nhau, tổng 16. Dùng độ dài `1+2+3+4 = 10`, còn 6 phải gộp vào đoạn dài nhất ⇒ `1,2,3,10` (4 đoạn)… hay `1,2,3,4,6` = 16 ⇒ **5 đoạn**. Tổng quát: lớn nhất `k` với `1+2+…+k ≤ n` (`k(k+1)/2 ≤ 16` ⇒ `k = 5`). Ô trên đã assert = 5.

In [17]:
random.seed(2024)
for _ in range(60):
    L = random.randint(1, 16)
    s = "".join(random.choice("abcd"[:random.randint(1, 4)]) for _ in range(L))
    want = max_split_bitmask(s)
    for f in ALL:
        assert f(s) == want, (f, s)
# công thức cho chuỗi một loại ký tự
for n in range(1, 17):
    k = 0
    while (k + 1) * (k + 2) // 2 <= n:
        k += 1
    assert max_split_pruned("z" * n) == k
print("Ngẫu nhiên + công thức chuỗi một ký tự: OK")

Ngẫu nhiên + công thức chuỗi một ký tự: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng tham lam "lấy đoạn mới ngắn nhất"

Ý tưởng sai: đi từ trái sang, tích lũy ký tự cho đến khi được một chuỗi chưa dùng thì cắt; phần dư cuối gộp vào đoạn trước.

In [18]:
def wrong_greedy(s):
    pieces, cur = [], ""
    for ch in s:
        cur += ch
        if cur not in pieces:
            pieces.append(cur)
            cur = ""
    if cur:                       # phần dư: gộp vào đoạn cuối
        pieces[-1] += cur
    return len(pieces) if len(set(pieces)) == len(pieces) else len(pieces) - 1

s = "aaba"
print("tham lam:", wrong_greedy(s), "| đúng:", Solution().maxUniqueSplit(s))

tham lam: 2 | đúng: 3


**Vì sao sai:** tham lam lấy `'a'`, rồi `'ab'`, còn dư `'a'` phải gộp ⇒ `['a', 'aba']` chỉ 2 đoạn. Lời giải tối ưu `['aa', 'b', 'a']` cần chọn đoạn đầu **dài hơn** để dành `'a'` cho cuối. Quyết định sớm ảnh hưởng tới tương lai ⇒ phải quay lui.

### Lỗi 2 — Quên hoàn tác (`remove`) khi quay lui

In [19]:
def wrong_no_undo(s):
    n = len(s); used = set(); best = 0
    def dfs(start, count):
        nonlocal best
        if start == n:
            best = max(best, count); return
        for end in range(start + 1, n + 1):
            piece = s[start:end]
            if piece not in used:
                used.add(piece)
                dfs(end, count + 1)
                # QUÊN used.remove(piece)
    dfs(0, 0)
    return best

s = "aaba"
print("không undo:", wrong_no_undo(s), "| đúng:", Solution().maxUniqueSplit(s))

không undo: 2 | đúng: 3


**Vì sao sai:** sau khi thử nhánh bắt đầu bằng `'a'`, chuỗi `'a'` (và `'ab'`, …) vẫn nằm trong `used`. Khi sang nhánh `'aa' | 'b' | 'a'`, đoạn `'a'` cuối bị coi là "đã dùng" dù nhánh hiện tại chưa dùng nó ⇒ bỏ sót đáp án 3.

### Lỗi 3 — Lấy "phần còn lại" làm đoạn cuối mà không kiểm tra trùng

Nhiều bạn tối ưu bằng cách: ở mỗi bước, cập nhật `best = count + 1` (coi phần còn lại là một đoạn), rồi mới thử cắt tiếp.

In [20]:
def wrong_last_unchecked(s):
    n = len(s); used = set(); best = 0
    def dfs(start, count):
        nonlocal best
        best = max(best, count + 1)          # SAI: s[start:] có thể đã dùng!
        for end in range(start + 1, n):
            piece = s[start:end]
            if piece not in used:
                used.add(piece); dfs(end, count + 1); used.remove(piece)
    dfs(0, 0)
    return best

for s in ["aa", "aba", "aaaa"]:
    print(s, "-> sai:", wrong_last_unchecked(s), "| đúng:", Solution().maxUniqueSplit(s))

aa -> sai: 2 | đúng: 1
aba -> sai: 3 | đúng: 2
aaaa -> sai: 3 | đúng: 2


**Vì sao sai:** với `"aa"`, chọn `'a'` rồi coi phần còn lại `'a'` là đoạn cuối ⇒ 2, nhưng `'a'` đã dùng. **Sửa:** chỉ tính khi `s[start:] not in used`, hoặc đơn giản dùng điều kiện dừng `start == n` như mục 9.

### Lỗi 4 — Biến `best` toàn cục không được reset giữa các test

LeetCode tạo **một** đối tượng `Solution` (hoặc chạy nhiều test trong cùng tiến trình). Nếu `best` là biến toàn cục / thuộc tính lớp không reset, kết quả test trước "rò" sang test sau.

In [21]:
class WrongSolution:
    best = 0                                # thuộc tính lớp: dùng chung mọi lần gọi!
    def maxUniqueSplit(self, s):
        n = len(s); used = set()
        def dfs(start, count):
            if start == n:
                WrongSolution.best = max(WrongSolution.best, count); return
            for end in range(start + 1, n + 1):
                piece = s[start:end]
                if piece not in used:
                    used.add(piece); dfs(end, count + 1); used.remove(piece)
        dfs(0, 0)
        return WrongSolution.best

w = WrongSolution()
print("test 1 'ababccc':", w.maxUniqueSplit("ababccc"))
print("test 2 'aa'     :", w.maxUniqueSplit("aa"), "| đúng:", Solution().maxUniqueSplit("aa"))

test 1 'ababccc': 5
test 2 'aa'     : 5 | đúng: 1


**Vì sao sai:** test 2 trả 5 (giá trị còn sót từ test 1). **Sửa:** khai báo `best` là biến cục bộ trong `maxUniqueSplit` và dùng `nonlocal`, như code nộp.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **`n ≤ 16` (hoặc ≤ 20)** + "chia/chọn" ⇒ gần như chắc chắn là **bitmask / quay lui**.
- "Các phần **khác nhau**" ⇒ `set` + add/remove trong quay lui.
- Bài **tối đa hóa** bằng quay lui ⇒ luôn nghĩ tới **cận trên** để cắt tỉa: "lạc quan nhất từ đây còn được bao nhiêu?".
- Thử lựa chọn "hứa hẹn" trước (đoạn ngắn trước) để `best` tăng sớm, cắt tỉa mạnh hơn.
- Nếu trạng thái tương lai phụ thuộc **cả tập** lựa chọn quá khứ ⇒ khó DP, hãy quay lui.

## 15. Biến thể của bài

1. **Trả về chính cách cắt** (danh sách đoạn), không chỉ số lượng.
2. **Đếm số cách cắt** thành các đoạn đôi một khác nhau.
3. **LC 131 — Palindrome Partitioning:** liệt kê mọi cách cắt mà mỗi đoạn là palindrome (quay lui tương tự, điều kiện khác).
4. **LC 139/140 — Word Break:** cắt sao cho mỗi đoạn thuộc từ điển (ở đây điều kiện "thuộc tập cho trước" thay vì "chưa dùng" ⇒ DP được).
5. **Tối thiểu số đoạn** sao cho các đoạn đôi một khác nhau — luôn là 1 (không cắt). Biến thể có ý nghĩa hơn: tối thiểu số đoạn mà **mỗi đoạn không có ký tự lặp** ⇒ tham lam (LC 2405).
6. **Cắt thành đúng `k` đoạn khác nhau** — có làm được không?

In [22]:
# Minh họa biến thể 2: đếm số cách cắt thành các đoạn đôi một khác nhau
def count_unique_splits(s):
    n = len(s); used = set()
    def dfs(start):
        if start == n:
            return 1
        total = 0
        for end in range(start + 1, n + 1):
            piece = s[start:end]
            if piece not in used:
                used.add(piece); total += dfs(end); used.remove(piece)
        return total
    return dfs(0)
print(count_unique_splits("aba"), count_unique_splits("aaba"))  # 3 cách hợp lệ cho "aba"

3 5


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bitmask | thử mọi tập vị trí cắt | O(n·2ⁿ) | O(n) | chuẩn đối chiếu, code không đệ quy |
| 2. Quay lui + set | dựng dần, bỏ đoạn trùng | O(n·2ⁿ) worst | O(n) | lời giải tự nhiên |
| 3. Quay lui + tỉa | thêm cận trên `count + (n − start)` | worst mũ, thực tế nhanh nhất | O(n) | **nộp bài** |
| 4. Bitmask theo popcount | thử nhiều lát cắt trước, gặp là dừng | O(n·2ⁿ) | O(2ⁿ) | khi đáp án thường lớn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Đoán `maxUniqueSplit` cho `"abc"`, `"aaa"`, `"abab"`, `"wwwzfvedwfvhsww"` (cái cuối có thể chạy để kiểm tra).

**Bài 2 (tính tay):** Với `s = "aaaaaa"` (6 chữ `a`), chỉ ra một cách cắt đạt tối đa và giải thích vì sao không thể hơn.

**Bài 3 (điền TODO):** Hoàn thiện `best_split(s)` trả về **một** danh sách đoạn đạt số lượng tối đa.

**Bài 4 (tự viết lại):** Viết lại lời giải bằng quay lui **không dùng `nonlocal`** (hàm `dfs` trả về số đoạn tối đa của phần còn lại, hoặc `-inf` nếu không thể).

**Bài 5 (mini challenge):** Viết `can_split_k(s, k)` trả `True` nếu có thể cắt `s` thành **đúng `k`** đoạn đôi một khác nhau.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_split(s):
    n = len(s)
    used, path = set(), []
    best_path = []
    def dfs(start):
        # TODO: nếu start == n và len(path) > len(best_path) thì lưu bản sao path
        # TODO: thử mọi piece = s[start:end] chưa dùng: add, path.append, dfs, pop, remove
        pass
    dfs(0)
    return best_path

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def max_split_return_style(s):
    # TODO
    pass

def can_split_k(s, k):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: chuỗi khác nhau hoàn toàn ⇒ `n`. Mức 2: `"aaa"` ⇒ độ dài các đoạn phải khác nhau. Mức 3: `"abc"→3`, `"aaa"→2`, `"abab"→3` (`a, b, ab`).

**Bài 2.** Mức 1: các đoạn chỉ khác nhau bởi độ dài. Mức 2: `1+2+3 = 6`. Mức 3: `a | aa | aaa` = 3; 4 đoạn cần tổng ≥ `1+2+3+4 = 10 > 6`.

**Bài 3.** Mức 1: giống Cách 2 + thêm `path`. Mức 2: lưu `best_path = path[:]` (bản sao!). Mức 3: thêm cắt tỉa `len(path) + (n − start) <= len(best_path)` để nhanh hơn.

**Bài 4.** Mức 1: `dfs(start)` trả số đoạn tối đa cho `s[start:]` với `used` hiện tại. Mức 2: `start == n` ⇒ trả 0. Mức 3: `res = max(res, 1 + dfs(end))` cho các piece hợp lệ; khởi tạo `res = -inf`.

**Bài 5.** Mức 1: quay lui có thêm tham số đếm. Mức 2: dừng nhánh khi `count > k` hoặc `count + (n − start) < k`. Mức 3: chú ý `k` có thể > tối đa ⇒ `False`; mọi `k` từ 1 tới tối đa có đạt được không? (thử và tự kiểm chứng).

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
ans1 = [Solution().maxUniqueSplit(x) for x in ["abc", "aaa", "abab", "wwwzfvedwfvhsww"]]
print("Bài 1:", ans1)
assert ans1[:3] == [3, 2, 3]
assert ans1[3] == max_split_bitmask("wwwzfvedwfvhsww")

# Bài 2
assert Solution().maxUniqueSplit("aaaaaa") == 3
print("Bài 2: a | aa | aaa -> 3")

Bài 1: [3, 2, 3, 11]
Bài 2: a | aa | aaa -> 3


In [26]:
# Bài 3
def best_split_sol(s):
    n = len(s)
    used, path = set(), []
    best_path = []
    def dfs(start):
        nonlocal best_path
        if len(path) + (n - start) <= len(best_path):
            return
        if start == n:
            best_path = path[:]
            return
        for end in range(start + 1, n + 1):
            piece = s[start:end]
            if piece not in used:
                used.add(piece); path.append(piece)
                dfs(end)
                path.pop(); used.remove(piece)
    dfs(0)
    return best_path

for x in ["ababccc", "aba", "aa", "aaba", "wwwzfvedwfvhsww"]:
    bp = best_split_sol(x)
    assert "".join(bp) == x and len(set(bp)) == len(bp) == Solution().maxUniqueSplit(x)
print("Bài 3: OK, ví dụ", best_split_sol("ababccc"))

Bài 3: OK, ví dụ ['a', 'b', 'ab', 'c', 'cc']


In [27]:
# Bài 4
def max_split_return_style_sol(s):
    n = len(s); used = set()
    def dfs(start):
        if start == n:
            return 0
        res = float("-inf")
        for end in range(start + 1, n + 1):
            piece = s[start:end]
            if piece not in used:
                used.add(piece)
                res = max(res, 1 + dfs(end))
                used.remove(piece)
        return res
    return dfs(0)

for L in range(1, 9):
    for tup in itertools.product("ab", repeat=L):
        x = "".join(tup)
        assert max_split_return_style_sol(x) == max_split_bitmask(x)
print("Bài 4: OK")

Bài 4: OK


In [28]:
# Bài 5
def can_split_k_sol(s, k):
    n = len(s); used = set()
    def dfs(start, count):
        if count > k or count + (n - start) < k:
            return False
        if start == n:
            return count == k
        for end in range(start + 1, n + 1):
            piece = s[start:end]
            if piece not in used:
                used.add(piece)
                ok = dfs(end, count + 1)
                used.remove(piece)
                if ok:
                    return True
        return False
    return dfs(0, 0)

def can_split_k_brute(s, k):
    n = len(s)
    for mask in range(1 << (n - 1)):
        parts = split_by_mask(s, mask)
        if len(parts) == k and len(set(parts)) == k:
            return True
    return False

random.seed(5)
for _ in range(300):
    x = "".join(random.choice("abc") for _ in range(random.randint(1, 9)))
    for k in range(1, len(x) + 2):
        assert can_split_k_sol(x, k) == can_split_k_brute(x, k)
print("Bài 5: OK;", [can_split_k_sol("ababccc", k) for k in range(1, 8)])

Bài 5: OK; [True, True, True, True, True, False, False]


Nhận xét từ Bài 5: với `"ababccc"`, mọi `k` từ 1 đến 5 đều đạt được, `k = 6, 7` thì không — ta đã kiểm chứng bằng vét cạn chứ không đoán.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template quay lui tối đa hóa có cắt tỉa:**

```python
best = 0
used = set()
def dfs(start, count):
    nonlocal best
    if count + (n - start) <= best: return     # cận trên
    if start == n: best = count; return
    for end in range(start + 1, n + 1):
        piece = s[start:end]
        if piece in used: continue
        used.add(piece)
        dfs(end, count + 1)
        used.remove(piece)                       # hoàn tác!
dfs(0, 0)
```

**Checklist:**
- [ ] `add` và `remove` đi đôi (hoàn tác đầy đủ)?
- [ ] Điều kiện dừng là `start == n` (đoạn cuối cũng được kiểm tra trùng)?
- [ ] `best` là biến cục bộ + `nonlocal`, không phải biến toàn cục/thuộc tính lớp?
- [ ] Cận trên cắt tỉa dùng `<=` (hoặc `<`) đúng chiều, không loại nhầm nhánh tốt hơn?
- [ ] Đã thử: `"a"`, chuỗi toàn một ký tự, chuỗi toàn ký tự khác nhau dài 16, `"aaba"`?